# Lab 13 — Jira Assistant RAG Agent (Step-by-Step)

This lab will guide you through building and running a CLI-based Retrieval-Augmented Generation (RAG) agent for Jira documentation using LangChain v1.x, OpenAI, and FAISS. Each step is runnable and explained.


## Step 1: Install Required Packages

Run this cell to ensure all dependencies are installed. This will install the packages listed in `requirements.txt` for this lab.


In [ ]:
# Step 1: Install Required Packages
!pip install langchain-openai langchain-community langchain-core langchain-classic python-dotenv faiss-cpu

## Step 2: Set Up Environment and Imports

Import all required libraries for the RAG agent.

In [ ]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import Tool
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_classic.chains import RetrievalQA
from dotenv import load_dotenv

## Step 3: Load Environment Variables

Load your OpenAI API key and check that it is available.

In [ ]:
# Load environment variables from .env file
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
print(f"[DEBUG] OpenAI API Key loaded: {'Yes' if openai_api_key else 'No'}")
if not openai_api_key:
    raise ValueError("OpenAI API key not found. Please set the OPENAI_API_KEY environment variable.")

## Step 4: Ingest Documentation and Build Vector Store

Load the Jira documentation, split it into chunks, and build the FAISS vector store.

In [ ]:
md_path = "ecommerce-Jira.md"
loader = TextLoader(md_path)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = text_splitter.split_documents(docs)

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(split_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

## Step 5: Set Up LLM and Prompt

Configure the OpenAI LLM and the prompt template for the agent.

In [ ]:
llm = ChatOpenAI(model="gpt-4", temperature=0, api_key=openai_api_key)
prompt_template = """
You are Jira Assistant, an expert in Jira project management and workflows. Use the ingested ecommerce-Jira.md documentation to answer user questions about Jira, its features, and best practices. If the answer is not in the documentation, say \"I don't know based on the Jira documentation.\"
Question: {input}
{agent_scratchpad}
"""

## Step 6: Create QA Chain and Tool

Set up the retrieval QA chain and the Jira search tool for the agent.

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=False
)

def jira_search_tool(query: str) -> str:
    result = qa_chain.invoke({"query": query})
    answer = str(result)
    print(f"[DEBUG] Jira search result: {answer}")
    if answer.strip() == "" or "I don't know" in answer:
        return "I don't know based on the Jira documentation."
    return answer

jira_tool = Tool(
    name="jira_search",
    func=jira_search_tool,
    description="Use this tool to answer Jira-related questions based on the ecommerce-Jira.md documentation. If the answer is not found, return 'I don't know based on the Jira documentation.' as a string."
)

## Step 7: Create Agent and Executor

Build the agent and executor to handle user questions.

In [ ]:
agent = create_tool_calling_agent(llm, [jira_tool], PromptTemplate.from_template(prompt_template))
executor = AgentExecutor(agent=agent, tools=[jira_tool])

## Step 8: Ask Questions (Interactively)

You can now ask Jira-related questions. Example:

In [ ]:
# Example: Run this cell to ask a question
question = "What are the acceptance criteria for Add to Cart?"
response = executor.invoke({"input": question})
print("Bot:", response.get("output", response))

<div style="background: #111; border-radius: 12px; border: 2px solid #ff9800; padding: 20px; display: flex; align-items: center; gap: 24px;">
    <img src="../images/robo1.png" alt="Robo1" style="width: 120px; height: auto; border-radius: 8px;">
    <div style="color: #fff;">
        <strong>Robo1 says:</strong><br>
        <span style="font-size:120%; color:#ff9800;">🧠 RAG Retriever Lab Complete!</span><br>
        You have connected your knowledge sources (Jira, TestRail, API contracts, and Postgres schema) into a single FAISS-backed retriever, exposed it through an MCP server, and used it in GitHub Copilot Chat for grounded QA analysis. You can now validate requirements, test coverage, API behavior, and database checks from one prompt-driven workflow.
    </div>
</div>